In [1]:
import numpy as np 
import pandas as pd
import vectorbt as vbt

from prices import load_prices
from kalman import kalman_beta
from signals import compute_spread_z, generate_signals

In [2]:
pairs = pd.read_csv("data/ah_pairs_arb2.csv")
pairs.head(15)

,company,A,H
0,MIDEA GROUP,000333.SZ,0300.HK
1,CATL,300750.SZ,3750.HK
2,HAIER SMARTHOME,600690.SS,6690.HK


In [3]:
results = []

for _, row in pairs.iterrows():
    print(row["company"])
    df = load_prices(row["A"], row["H"])
    if df is None or len(df) < 500:
        continue

    df = df.loc["2021-01-01":]   
    
    A = df["A_Close"]
    H = df["H_Close"]

    beta = kalman_beta(A, H)
    
    spread, z = compute_spread_z(A, H, beta)
    entry, exit = 2.0, 0.5
    entries, exits, direction = generate_signals(z,entry=entry,exit=exit)
    
    long_entries  = z < -entry
    short_entries = z > entry

    long_exits  = z > -exit
    short_exits = z < exit
    hard_stop = 3.5

    forced_long_exit = z.abs() > hard_stop
    forced_short_exit = z.abs() < -hard_stop
    final_long_exits = long_exits | forced_long_exit
    final_short_exits = short_exits | forced_short_exit

    if (long_entries | short_entries).sum() == 0:
        continue

    pf = vbt.Portfolio.from_signals(
        close=A,
        entries=long_entries,
        exits=final_long_exits,
        short_entries=short_entries,
        short_exits=final_short_exits,
        init_cash=1_000_000,
        fees=0.008,
        freq="1D"
    )

    pos = pf.assets().shift(1).fillna(0)
    spread_pnl = pos * (A.diff() - beta.shift(1) * H.diff())

    spread_pnl = spread_pnl.fillna(0)
    
    if spread_pnl.std() == 0:
        continue

    sharpe = spread_pnl.mean() / spread_pnl.std() * np.sqrt(252)

    equity = spread_pnl.cumsum()
    max_dd = (equity - equity.cummax()).min()

    results.append({
        "company": row["company"],
        "A": row["A"],
        "H": row["H"],
        "Trades": entries.sum(),
        "Sharpe": sharpe,
        "Max_Drawdown": max_dd,
        "Total_PnL": equity.iloc[-1]
    })

MIDEA GROUP
YF.download() has changed argument auto_adjust default to True
CATL
HAIER SMARTHOME


In [4]:
res_df = pd.DataFrame(results)
print(res_df.shape)
res_df.sort_values("Sharpe", ascending=False)
res_df.head(15)

(1, 7)


,company,A,H,Trades,Sharpe,Max_Drawdown,Total_PnL
0,HAIER SMARTHOME,600690.SS,6690.HK,94,1.900172,-55923.855247,1.065964e+06


In [5]:
print("Pairs tested:", len(pairs))
print("Pairs traded:", len(res_df))
res_df[["company", "Trades", "Sharpe", "Max_Drawdown"]]

Pairs tested: 3
Pairs traded: 1


,company,Trades,Sharpe,Max_Drawdown
0,HAIER SMARTHOME,94,1.900172,-55923.855247
